# Natural Language Processing Lab - Introduction
## Task 1: Multimodal Data Ingestion, Inspection, and Preprocessing

**Objective**:
Demonstrate proficiency in handling, parsing, and inspecting diverse data formats routinely encountered in NLP and multimodal machine learning workflows:
- Unstructured text (`.txt`)
- Compressed tabular data (`.csv` / `.zip`)
- Semi-structured JSON (`.json`)
- Image data (`.jpg` / `.png`)
- Audio signals (`.mp3`)
- Video streams (`.mp4`)
- Spreadsheet workbooks (`.xlsx`)

In [ ]:
import os
import zipfile
import json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import IPython.display as ipd

# Base directory for datasets
DATA_DIR = Path("../data")
print(f"Configured dataset root: {DATA_DIR.resolve()}")

### 1. Plain Text Corpus Ingestion (`.txt`)
We inspect the Project Gutenberg literature text file (`1342-0.txt`), calculating total lines, words, character count, and previewing the header content.

In [ ]:
text_filepath = DATA_DIR / "text" / "1342-0.txt"

def inspect_text_file(filepath: Path, preview_lines: int = 12):
    with open(filepath, "r", encoding="utf-8") as file_handle:
        raw_lines = file_handle.readlines()
        
    full_content = "".join(raw_lines)
    word_tokens = full_content.split()
    
    print("=" * 60)
    print(f"TEXT FILE ANALYSIS: {filepath.name}")
    print("=" * 60)
    print(f"Total Lines      : {len(raw_lines):,}")
    print(f"Total Words      : {len(word_tokens):,}")
    print(f"Total Characters : {len(full_content):,}")
    print("-" * 60)
    print(f"First {preview_lines} Lines Preview:")
    for idx, line in enumerate(raw_lines[:preview_lines], 1):
        print(f"[{idx:02d}] {line.strip()}")

inspect_text_file(text_filepath)

### 2. Compressed Tabular Data Extraction (`.csv.zip`)
We extract and inspect the Fisher Iris dataset directly from an archived ZIP format.

In [ ]:
zip_filepath = DATA_DIR / "csv" / "iris.csv.zip"

def inspect_zipped_csv(zip_path: Path):
    with zipfile.ZipFile(zip_path, "r") as archive:
        file_list = archive.namelist()
        print(f"Archive contents: {file_list}")
        
        # Target iris data file within archive
        target_csv = [f for f in file_list if f.endswith("iris.data")][0]
        with archive.open(target_csv) as csv_stream:
            column_headers = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
            iris_df = pd.read_csv(csv_stream, header=None, names=column_headers)
            
    print("\nIris Dataset Overview:")
    print(f"Shape: {iris_df.shape[0]} rows x {iris_df.shape[1]} columns")
    print("\nClass Distribution:")
    print(iris_df["species"].value_counts())
    print("\nDescriptive Statistics:")
    display(iris_df.describe().round(2))
    return iris_df

iris_dataframe = inspect_zipped_csv(zip_filepath)

### 3. Semi-Structured Data Parsing (`.json`)
Parsing nested JSON records (`restaurant.json`) and transforming the payload into tabular representation.

In [ ]:
json_filepath = DATA_DIR / "json" / "restaurant.json"

def inspect_json_records(filepath: Path):
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line_str = line.strip()
            if line_str:
                try:
                    records.append(json.loads(line_str))
                except json.JSONDecodeError:
                    pass
    if not records:
        with open(filepath, "r", encoding="utf-8") as f:
            payload = json.load(f)
            records = payload if isinstance(payload, list) else [payload]
            
    print(f"Loaded {len(records):,} records from {filepath.name}")
    if records:
        print(f"Sample keys : {list(records[0].keys())}")
        df_json = pd.json_normalize(records)
    else:
        df_json = pd.DataFrame()
        
    print("\nNormalized JSON DataFrame Preview:")
    display(df_json.head(4))
    return df_json

restaurant_df = inspect_json_records(json_filepath)

### 4. Image Asset Loading & Inspection (`.jpg` / `.png`)
Loading image assets using PIL, analyzing dimensions, color channels, and tensor representations.

In [ ]:
image_filepath = DATA_DIR / "images" / "convertico-ninja-girl-jpg.jpg"

def inspect_image_asset(img_path: Path):
    img = Image.open(img_path)
    img_array = np.array(img)
    
    print("=" * 50)
    print("IMAGE METADATA")
    print("=" * 50)
    print(f"File Name   : {img_path.name}")
    print(f"Dimensions  : {img.width} x {img.height} pixels")
    print(f"Color Mode  : {img.mode}")
    print(f"Array Shape : {img_array.shape}")
    print(f"Pixel Range : Min={img_array.min()}, Max={img_array.max()}")
    
    display(img)

inspect_image_asset(image_filepath)

### 5. Audio Media Inspection (`.mp3`)
Verifying audio file attributes and rendering the interactive audio player widget.

In [ ]:
audio_filepath = DATA_DIR / "audio" / "tone-test.mp3"

def inspect_audio_asset(aud_path: Path):
    file_size_kb = aud_path.stat().st_size / 1024
    print(f"Audio File : {aud_path.name}")
    print(f"File Size  : {file_size_kb:.2f} KB")
    
    # Interactive playback widget
    display(ipd.Audio(str(aud_path)))

inspect_audio_asset(audio_filepath)

### 6. Video Stream Inspection (`.mp4`)
Verifying video assets and configuring playback inside the notebook.

In [ ]:
video_filepath = DATA_DIR / "video" / "video.mp4"

def inspect_video_asset(vid_path: Path):
    file_size_mb = vid_path.stat().st_size / (1024 * 1024)
    print(f"Video File : {vid_path.name}")
    print(f"File Size  : {file_size_mb:.2f} MB")
    
    display(ipd.Video(str(vid_path), embed=True, width=420))

inspect_video_asset(video_filepath)

### 7. Spreadsheet Data Loading (`.xlsx`)
Ingesting enterprise transaction spreadsheets using Pandas and OpenPyXL engine.

In [ ]:
excel_filepath = DATA_DIR / "xlsx" / "Online Retail.xlsx"

def inspect_excel_workbook(xlsx_path: Path, sample_limit: int = 5000):
    excel_df = pd.read_excel(xlsx_path, nrows=sample_limit)
    print(f"Sampled Shape : {excel_df.shape[0]} rows x {excel_df.shape[1]} columns")
    print("\nColumn Data Types:")
    print(excel_df.dtypes)
    print("\nTransaction Preview:")
    display(excel_df.head(5))
    return excel_df

retail_df = inspect_excel_workbook(excel_filepath)

### Summary of Ingested Formats

| Format | Source File | Key Python Library | Primary Output Structure |
| :--- | :--- | :--- | :--- |
| **Plain Text** | `1342-0.txt` | Built-in / `io` | String / Token List |
| **Zipped CSV** | `iris.csv.zip` | `zipfile`, `pandas` | 2D DataFrame (150x5) |
| **JSON** | `restaurant.json` | `json`, `pandas` | Normalized Tabular DataFrame |
| **Raster Image**| `convertico-ninja-girl-jpg.jpg` | `PIL.Image`, `numpy` | 3D NumPy Array (H x W x C) |
| **Audio** | `tone-test.mp3` | `IPython.display` | Interactive Audio Stream |
| **Video** | `video.mp4` | `IPython.display` | Interactive Video Player |
| **Spreadsheet**| `Online Retail.xlsx` | `openpyxl`, `pandas` | Tabular Transaction Matrix |